In [3]:
from pathlib import Path
import json, pandas as pd, numpy as np

DATA = Path('./data/')

schema = json.loads((DATA / 'feature_schema_context.json').read_text(encoding='utf-8'))
df = pd.read_parquet(DATA / 'model_features_context.parquet')
print(df.shape)                            # (2629440, 56)

# pd.set_option("display.max_columns", None)
# df
# 
INPUTS = schema['inputs']          # 입력 39개 (직접 고르지 말고 이 목록을 쓰기)
TARGET = 'rentals_target_1h'       # 반납 모델은 'returns_target_1h'

use = df.coverage_status.eq('observed_operating') & df[TARGET].notna()
train = df[use & df.split.eq('train')]
valid = df[use & df.split.eq('validation')]
test  = df[use & df.split.eq('test')]      # 마지막까지 열어보지 않기
print(len(train), len(valid), len(test))

(2629440, 56)
1569378 526178 524875


In [6]:
import json
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import joblib

print("🧠 Executing Self-Contained GRU Deep Learning Pipeline...")

# MODULE INITIALIZATION & MAPPING
INPUTS = schema['inputs']
TARGETS = ['rentals_target_1h', 'returns_target_1h']

# Step 2: Time-series masks extraction
use = df.coverage_status.eq('observed_operating') & df['rentals_target_1h'].notna() & df['returns_target_1h'].notna()
train_mask = use & df.split.eq('train')
valid_mask = use & df.split.eq('validation')
test_mask  = use & df.split.eq('test')

# Generate isolated coordinate points to align subsets accurately
X_valid_idx = df[valid_mask].index
X_test_idx  = df[test_mask].index

# Core evaluation score matrix
def score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None) # Clip negative numbers to 0
    err = y_pred - y_true
    total = np.abs(y_true).sum()
    
    mae = np.abs(err).mean()
    rmse = np.sqrt((err**2).mean())
    wape = np.abs(err).sum() / total if total else np.nan
    bias = err.mean()
    
    return {'MAE': mae, 'RMSE': rmse, 'WAPE': wape, 'Bias': bias}

# Explicit segment definitions mapped directly to your dataset labels
segments = {
    '전체': (X_valid_idx, X_test_idx),
    
    '출근': (X_valid_idx[df.loc[X_valid_idx, 'commute_period'] == 'morning'], 
            X_test_idx[df.loc[X_test_idx, 'commute_period'] == 'morning']),
    
    '퇴근': (X_valid_idx[df.loc[X_valid_idx, 'commute_period'] == 'evening'], 
            X_test_idx[df.loc[X_test_idx, 'commute_period'] == 'evening']),
    
    '역세권': (X_valid_idx[df.loc[X_valid_idx, 'station_group'] == '역세권'], 
              X_test_idx[df.loc[X_test_idx, 'station_group'] == '역세권']),
    
    '업무지구': (X_valid_idx[df.loc[X_valid_idx, 'station_group'] == '업무지구'], 
                X_test_idx[df.loc[X_test_idx, 'station_group'] == '업무지구']),
    
    '주거지': (X_valid_idx[df.loc[X_valid_idx, 'station_group'] == '주거지'], 
              X_test_idx[df.loc[X_test_idx, 'station_group'] == '주거지'])
}

metrics_records = []
pred_df_list = []

# TYPE-SAFE DATA PREPROCESSING FOR NEURAL NETWORKS
print("🔧 Preprocessing categorical and boolean mappings for deep layers...")
X_copy = df[INPUTS].copy()
cat_cols = ['station_group', 'region', 'commute_period', 'holiday_type']
num_cols = [c for c in INPUTS if c not in cat_cols]

# 1. Safely force any boolean columns into binary 1s and 0s first
for col in num_cols:
    if pd.api.types.is_bool_dtype(X_copy[col]):
        # Converts True -> 1, False -> 0, keeps NaNs as NaN securely
        X_copy[col] = X_copy[col].astype('float64')

# 2. Fill remaining numeric observations with 0 to prevent network fragmentation
for col in num_cols:
    X_copy[col] = X_copy[col].fillna(0.0).astype('float64')

# 3. Map string categories to numerical sequences natively
cat_mappings = {}
for col in cat_cols:
    X_copy[col] = X_copy[col].astype(str).fillna('missing')
    X_copy[col], uniques = pd.factorize(X_copy[col])
    cat_mappings[col] = len(uniques)

# Isolate feature dataframes using masks
X_train, X_valid, X_test = X_copy[train_mask], X_copy[valid_mask], X_copy[test_mask]

def prep_input_dict(X_matrix):
    d = {f"input_{c}": X_matrix[c].values for c in cat_cols}
    d["input_numeric"] = X_matrix[num_cols].values
    return d

# Helper function to compile Functional GRU layers
def build_gru_model(numeric_dim, cat_dims):
    inputs_list = []
    embeddings = []
    
    for col_name, num_classes in cat_dims.items():
        inp = layers.Input(shape=(1,), name=f"input_{col_name}")
        inputs_list.append(inp)
        emb = layers.Embedding(input_dim=num_classes + 1, output_dim=8)(inp)
        emb = layers.Flatten()(emb)
        embeddings.append(emb)
        
    num_inp = layers.Input(shape=(numeric_dim,), name="input_numeric")
    inputs_list.append(num_inp)
    
    merged = layers.concatenate(embeddings + [num_inp])
    reshape_layer = layers.Reshape((1, merged.shape[-1]))(merged)
    
    gru_out = layers.GRU(64, return_sequences=False)(reshape_layer)
    dense_out = layers.Dense(32, activation='relu')(gru_out)
    output = layers.Dense(1, activation='linear')(dense_out)
    
    model = models.Model(inputs=inputs_list, outputs=output)
    model.compile(optimizer=tf.keras.optimizers.Adam(0.005), loss='mae')
    return model

# MODEL TRAINING & PROCESSING LOOPS
for target in TARGETS:
    print(f"\n🧠 Training GRU Model for target variable: {target}...")
    
    y_train = df.loc[train_mask, target]
    y_valid = df.loc[valid_mask, target]
    y_test  = df.loc[test_mask, target]

    tf.keras.utils.set_random_seed(42)
    model = build_gru_model(len(num_cols), cat_mappings)
    
    start_time = time.time()
    # Large batch size chosen to process the massive tabular frame rapidly
    model.fit(
        prep_input_dict(X_train), y_train.values,
        validation_data=(prep_input_dict(X_valid), y_valid.values),
        epochs=10, 
        batch_size=2048,
        callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        verbose=0
    )
    train_seconds = time.time() - start_time
    print(f"⏱️ Model training completed in {train_seconds:.2f} seconds.")
    model.save(f'./models/bicycle_gru_{target}.keras')
    
    # Generate batch predictions
    valid_pred_arr = np.clip(model.predict(prep_input_dict(X_valid), batch_size=4096).flatten(), 0, None)
    test_pred_arr  = np.clip(model.predict(prep_input_dict(X_test), batch_size=4096).flatten(), 0, None)
    
    valid_pred_series = pd.Series(valid_pred_arr, index=X_valid.index)
    test_pred_series  = pd.Series(test_pred_arr, index=X_test.index)
    
    # Run evaluation across segments
    for seg_name, (v_idx, t_idx) in segments.items():
        v_res = score(df.loc[v_idx, target], valid_pred_series.loc[v_idx])
        t_res = score(df.loc[t_idx, target], test_pred_series.loc[t_idx])
        
        metrics_records.append({
            'model': 'GRU', 'target': target, 'split': 'validation', 'segment': seg_name,
            'n': len(v_idx), 'MAE': v_res['MAE'], 'RMSE': v_res['RMSE'],
            'WAPE': v_res['WAPE'], 'Bias': v_res['Bias'], 'train_seconds': train_seconds
        })
        metrics_records.append({
            'model': 'GRU', 'target': target, 'split': 'test', 'segment': seg_name,
            'n': len(t_idx), 'MAE': t_res['MAE'], 'RMSE': t_res['RMSE'],
            'WAPE': t_res['WAPE'], 'Bias': t_res['Bias'], 'train_seconds': train_seconds
        })

    # Restructure prediction row logs for Parquet file output
    for split_name, mask, pred_vals in [('validation', valid_mask, valid_pred_arr), ('test', test_mask, test_pred_arr)]:
        sub_pred = df.loc[mask, ['station_id', 'timestamp']].copy()
        sub_pred['split'] = split_name
        sub_pred['target'] = target
        sub_pred['y_true'] = df.loc[mask, target].astype(float)
        sub_pred['y_pred'] = pred_vals.astype(float)
        pred_df_list.append(sub_pred)

# STEP 5: OUTPUT DELIVERY FILE WRITING
print("\n💾 Writing production outputs to Shared Drive directory path...")

DATA = Path('./results/')

metrics_df = pd.DataFrame(metrics_records)
metrics_df.to_csv(DATA / 'metrics_GRU.csv', index=False)

pred_df = pd.concat(pred_df_list, ignore_index=True)
pred_df.to_parquet(DATA / 'pred_GRU.parquet', index=False, engine='pyarrow')

params_dict = {
    "inputs": INPUTS,
    "hyperparameters": {
        "architecture": "Tabular GRU",
        "optimizer": "Adam",
        "learning_rate": 0.005,
        "gru_units": 64,
        "dense_units": 32
    }
}
with open(DATA / 'params_GRU.json', 'w', encoding='utf-8') as f:
    json.dump(params_dict, f, indent=2, ensure_ascii=False)

print("🎯 Finished successfully! GRU deep learning model artifacts built completely.")


🧠 Executing Self-Contained GRU Deep Learning Pipeline...
🔧 Preprocessing categorical and boolean mappings for deep layers...

🧠 Training GRU Model for target variable: rentals_target_1h...
⏱️ Model training completed in 123.44 seconds.
129/129 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step
129/129 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step

🧠 Training GRU Model for target variable: returns_target_1h...
⏱️ Model training completed in 147.39 seconds.
129/129 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step
129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step

💾 Writing production outputs to Shared Drive directory path...
🎯 Finished successfully! GRU deep learning model artifacts built completely.


In [4]:
import pandas as pd

# 1. Load the generated metrics file from your DATA path
metrics_path = DATA / 'metrics_GRU.csv'
metrics_df = pd.read_csv(metrics_path)

# 2. Separate Validation and Test for clean viewing
val_summary = metrics_df[metrics_df['split'] == 'validation'].copy()
test_summary = metrics_df[metrics_df['split'] == 'test'].copy()

# 3. Display Validation Dashboard (Use this to compare against Team A's Baseline)
print("=================== 📊 GRU 검증 ===================")
val_pivot = val_summary.pivot(index='segment', columns='target', values=['MAE', 'WAPE', 'Bias'])
display(val_pivot.round(4))

# 4. Display Final Test Dashboard (The 2025 Evaluation - Checked Once)
print("\n=================== 🏁 GRU 테스트 ===================")
test_pivot = test_summary.pivot(index='segment', columns='target', values=['MAE', 'WAPE', 'Bias'])
display(test_pivot.round(4))


=================== 📊 GRU 검증 ===================


MAE                                WAPE  \
target  rentals_target_1h returns_target_1h rentals_target_1h   
segment                                                         
업무지구               1.4795            1.4625            0.5125   
역세권                1.9066            1.9229            0.4381   
전체                 1.5552            1.5429            0.4910   
주거지                1.2792            1.2431            0.5655   
출근                 1.9977            2.0704            0.4709   
퇴근                 2.4997            2.4312            0.4130   

                                       Bias                    
target  returns_target_1h rentals_target_1h returns_target_1h  
segment                                                        
업무지구               0.4822           -0.4512           -0.5599  
역세권                0.4312           -0.3310           -0.6009  
전체                 0.4731           -0.4360           -0.5541  
주거지                0.5426           -0.5259           -0.5016  
출근                 0.4159           -0.5436           -0.7283  
퇴근                 0.4048           -0.4078           -0.8096


=================== 🏁 GRU 테스트 ===================


MAE                                WAPE  \
target  rentals_target_1h returns_target_1h rentals_target_1h   
segment                                                         
업무지구               1.2858            1.2680            0.5370   
역세권                1.6531            1.6614            0.4716   
전체                 1.3579            1.3411            0.5218   
주거지                1.1354            1.0944            0.5947   
출근                 1.7580            1.8219            0.4835   
퇴근                 2.1809            2.0581            0.4463   

                                       Bias                    
target  returns_target_1h rentals_target_1h returns_target_1h  
segment                                                        
업무지구               0.5016           -0.3511           -0.4296  
역세권                0.4589           -0.2218           -0.4904  
전체                 0.4989           -0.3288           -0.4427  
주거지                0.5709           -0.4134           -0.4081  
출근                 0.4180           -0.3576           -0.5946  
퇴근                 0.4349           -0.2455           -0.5597

In [3]:
cols = INPUTS + ['station_id', 'timestamp', 'nearest_subway',
                'rentals_target_1h', 'returns_target_1h']
app_df = df.loc[df.split.eq('test') & df.coverage_status.eq('observed_operating'), cols]
app_df = app_df[app_df.timestamp >= app_df.timestamp.max() - pd.Timedelta(days=28)]
app_df.to_parquet('./data/app_slice.parquet', index=False)

In [4]:
import joblib
import pandas as pd
from pathlib import Path

Path('./models').mkdir(exist_ok=True)

INPUTS = schema['inputs']
cat_cols = ['station_group', 'region', 'commute_period', 'holiday_type']

use = (df.coverage_status.eq('observed_operating')
       & df['rentals_target_1h'].notna()
       & df['returns_target_1h'].notna())
train_mask = use & df.split.eq('train')

# 1) feature_meta.pkl, used by LightGBM and XGBoost (one shared file)
feature_meta = {
    'inputs': INPUTS,
    'categories': {c: df.loc[train_mask, c].astype('category').cat.categories.tolist()
                   for c in cat_cols},
}
joblib.dump(feature_meta, './models/feature_meta.pkl')

# 2) gru_meta.pkl, used by the GRU
num_cols = [c for c in INPUTS if c not in cat_cols]
cat_maps = {}
for col in cat_cols:
    s = df[col].astype(str).fillna('missing')
    codes, uniques = pd.factorize(s)
    cat_maps[col] = {v: i for i, v in enumerate(uniques)}
joblib.dump({'num_cols': num_cols, 'cat_cols': cat_cols, 'cat_maps': cat_maps},
            './models/gru_meta.pkl')

# Quick check
for c in cat_cols:
    print(c, feature_meta['categories'][c], '| GRU codes:', len(cat_maps[c]))

station_group ['업무지구', '역세권', '주거지'] | GRU codes: 3
region ['강남구', '강동구', '강서구', '관악구', '광진구', '구로구', '금천구', '노원구', '도봉구', '동대문구', '마포구', '서대문구', '서초구', '성동구', '성북구', '송파구', '양천구', '영등포구', '용산구', '은평구', '종로구', '중구', '중랑구'] | GRU codes: 24
commute_period ['evening', 'morning', 'other'] | GRU codes: 3
holiday_type ['election', 'statutory', 'substitute', 'temporary'] | GRU codes: 5
